In [10]:
req.libs <- c("data.table", "dplyr", "ggplot2", "scattermore", 
              "stringi", "tidyr", "viridis", "Matrix", "tibble", "ComplexHeatmap", "circlize")
lapply(req.libs, function(x){
    suppressMessages(library(x, character.only = TRUE))})

[[1]]
[1] "data.table" "stats"      "graphics"   "grDevices"  "utils"     
[6] "datasets"   "methods"    "base"      

[[2]]
[1] "dplyr"      "data.table" "stats"      "graphics"   "grDevices" 
[6] "utils"      "datasets"   "methods"    "base"      

[[3]]
 [1] "ggplot2"    "dplyr"      "data.table" "stats"      "graphics"  
 [6] "grDevices"  "utils"      "datasets"   "methods"    "base"      

[[4]]
 [1] "scattermore" "ggplot2"     "dplyr"       "data.table"  "stats"      
 [6] "graphics"    "grDevices"   "utils"       "datasets"    "methods"    
[11] "base"       

[[5]]
 [1] "stringi"     "scattermore" "ggplot2"     "dplyr"       "data.table" 
 [6] "stats"       "graphics"    "grDevices"   "utils"       "datasets"   
[11] "methods"     "base"       

[[6]]
 [1] "tidyr"       "stringi"     "scattermore" "ggplot2"     "dplyr"      
 [6] "data.table"  "stats"       "graphics"    "grDevices"   "utils"      
[11] "datasets"    "methods"     "base"       

[[7]]
 [1] "viridis"     "viridisLite" "tidyr"       "stringi"     "scattermore"
 [6] "ggplot2"     "dplyr"       "data.table"  "stats"       "graphics"   
[11] "grDevices"   "utils"       "datasets"    "methods"     "base"       

[[8]]
 [1] "Matrix"      "viridis"     "viridisLite" "tidyr"       "stringi"    
 [6] "scattermore" "ggplot2"     "dplyr"       "data.table"  "stats"      
[11] "graphics"    "grDevices"   "utils"       "datasets"    "methods"    
[16] "base"       

[[9]]
 [1] "tibble"      "Matrix"      "viridis"     "viridisLite" "tidyr"      
 [6] "stringi"     "scattermore" "ggplot2"     "dplyr"       "data.table" 
[11] "stats"       "graphics"    "grDevices"   "utils"       "datasets"   
[16] "methods"     "base"       

[[10]]
 [1] "ComplexHeatmap" "grid"           "tibble"         "Matrix"        
 [5] "viridis"        "viridisLite"    "tidyr"          "stringi"       
 [9] "scattermore"    "ggplot2"        "dplyr"          "data.table"    
[13] "stats"          "graphics"       "grDevices"      "utils"         
[17] "datasets"       "methods"        "base"          

[[11]]
 [1] "circlize"       "ComplexHeatmap" "grid"           "tibble"        
 [5] "Matrix"         "viridis"        "viridisLite"    "tidyr"         
 [9] "stringi"        "scattermore"    "ggplot2"        "dplyr"         
[13] "data.table"     "stats"          "graphics"       "grDevices"     
[17] "utils"          "datasets"       "methods"        "base"

In [11]:
meta.df <- fread("/mnt/users/rachelsavage/paper_minimorf/data/frequent_files/meta.csv")

In [16]:
atac.dir <- "/fab/AlignedData/rsav_minimorf/igvf_pipeline/"
frag_files <- list.files(
  path = atac.dir,
  pattern = ".fragments.tsv.gz$",  # regex for exact match
  recursive = TRUE,                         # search subdirectories
  full.names = TRUE                         # return full paths
)

In [88]:
outdir <- "/fab/AlignedData/rsav_minimorf/splitting_fragments/filtered_fragments/"

In [17]:
head(frag_files)

[1] "/fab/AlignedData/rsav_minimorf/igvf_pipeline//SS-PKR-608-SS-V1-ENTIRE-PLATE/SS-PKR-608-SS-V1-ENTIRE-PLATE.fragments.tsv.gz"
[2] "/fab/AlignedData/rsav_minimorf/igvf_pipeline//SS-PKR-609-SS-V1-ENTIRE-PLATE/SS-PKR-609-SS-V1-ENTIRE-PLATE.fragments.tsv.gz"
[3] "/fab/AlignedData/rsav_minimorf/igvf_pipeline//SS-PKR-610-SS-V1-ENTIRE-PLATE/SS-PKR-610-SS-V1-ENTIRE-PLATE.fragments.tsv.gz"
[4] "/fab/AlignedData/rsav_minimorf/igvf_pipeline//SS-PKR-611-SS-V1-ENTIRE-PLATE/SS-PKR-611-SS-V1-ENTIRE-PLATE.fragments.tsv.gz"
[5] "/fab/AlignedData/rsav_minimorf/igvf_pipeline//SS-PKR-612-SS-V1-ENTIRE-PLATE/SS-PKR-612-SS-V1-ENTIRE-PLATE.fragments.tsv.gz"
[6] "/fab/AlignedData/rsav_minimorf/igvf_pipeline//SS-PKR-613-SS-V1-ENTIRE-PLATE/SS-PKR-613-SS-V1-ENTIRE-PLATE.fragments.tsv.gz"

In [30]:
#create combos of condition, tf, etc

# meta.df %>% filter(n.tfs == 1) %>% 
# group_by(final.tf) %>% 
# summarize(n.reads = sum(ATAC_unique_mapped_reads))%>%
# arrange(n.reads) %>% head()

# ##when grouping by TF, even the smallest TF has a lotta reads 

# meta.df %>% filter(n.tfs == 1) %>% 
# group_by(final.tf,condition) %>% 
# summarize(n.reads = sum(ATAC_unique_mapped_reads))%>%
# arrange(n.reads) %>% head()

#### 30,000 unique fragments per cell type/cluster for meaningful peak calling in aggregated (pseudo-bulk) data

##so i can be as fine grained as i want! sweet. 

#let's do:
#all
#condition
#tf
#tf x condition 
df1 <- meta.df[,c("barcode", "sublibrary")] %>%
mutate(splitcol = "All")
df2 <- meta.df[,c("barcode", "sublibrary", "condition")] %>%
rename(splitcol = condition)
df3 <- meta.df %>%
filter(n.tfs == 1) %>% 
dplyr::select(barcode, sublibrary, final.tf) %>% 
rename(splitcol = final.tf)

df4 <- meta.df %>%
filter(n.tfs == 1) %>% 
mutate(splitcol = paste0(final.tf, "_", condition))%>%
dplyr::select(barcode, sublibrary, splitcol)

combined <- rbindlist(list(df1, df2, df3, df4), use.names = TRUE, fill = TRUE)
dt_list <- split(combined, by = "sublibrary", keep.by = TRUE)

In [ ]:
filter.frags <- function(x){
    frag <- fread(grep(names(dt_list)[[x]], frag_files,value = TRUE))
    meta.input <- dt_list[[names(dt_list)[[x]]]]
    meta_list <- split(meta.input, by = "splitcol", keep.by = TRUE)
    
    lapply(names(meta_list), function(split){
    filtered_frags <- frag[V4 %in% meta_list[[split]]$barcode]
    filename <- paste0(outdir, "/", split, "_", names(dt_list)[[x]], "_fragments.tsv")
    fwrite(filtered_frags, file = filename, sep = "\t", col.names = FALSE)
    })

    return(message(Sys.time(), " finished ", names(dt_list)[[x]]))
}

In [75]:
#pbmclapply(1:length(dt_list), filter.frags,mc.cores=10)


In [ ]:
all.files <- list.files(outdir)
common.files <- stri_split_fixed(all.files, pattern = "_SS-PKR", simplify = TRUE)[,1]
file.df <- data.frame(fullname = all.files, short = common.files)

In [ ]:
concat.fx <- function(x){
	input.files <- file.df %>% filter(short == x)
	output_file <- paste0("/fab/AlignedData/rsav_minimorf/splitting_fragments/combined/", x, ".frag.tsv")
	filelist <- paste(shQuote(input.files$path), collapse = " ")
	cmd <- sprintf("cat %s > %s", filelist, shQuote(output_file))
	system(cmd)
}


In [ ]:
#pbmclapply(unique(file.df$short), concat.fx,mc.cores=10)


In [ ]:
peakdir <- "/fab/AlignedData/rsav_minimorf/splitting_fragments/peaks/"
files <- list.files(path = "/fab/AlignedData/rsav_minimorf/splitting_fragments/combined/", pattern = "*.frag.tsv", full.names = TRUE)

call.peaks <- function(x){
name <- sub("\\.frag\\.tsv$", "", basename(x))
cmd <- sprintf("macs2 callpeak --nomodel -t %s -n %s -f BEDPE --nolambda --keep-dup all --call-summits --outdir %s", shQuote(x), shQuote(name), shQuote(peakdir))
system(cmd)
return(message("finished ", x, " ", Sys.time()))
}


In [ ]:
#lapply(files, call.peaks)